In [1]:
!pip install sentence-transformers chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 1.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.8/20.8 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.0/208.0 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 7.0 MB/s eta 

In [8]:
import chromadb
from chromadb import PersistentClient    # DB 접속자 역할
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

embedding_fn = SentenceTransformerEmbeddingFunction(model_name='all-MiniLM-L6-v2')
client = PersistentClient(path='./chroma_db')

# 컬랙션 생성
# client.delete_collection(name='test')
collection = client.get_or_create_collection(name='test', embedding_function=embedding_fn)

# 컬랙션에 자료 추가(add - insert)
collection.add(
    documents=[
        "문서1:금요일 퇴근 후에 헬스장",
        "문서2:잠은 언제 자나~"
    ],
    metadatas=[
        {'tag': 'mes1'},
        {'tag': 'mes2'}
    ],
    ids=["doc1", "doc2"]
)

# 자료 조회 get - (sql)select
results = collection.get(include=['documents','metadatas','embeddings'])
for doc, meta, id, emb in zip(results['documents'], results['metadatas'], results['ids'], results['embeddings']):
    print(f"id : {id}")
    print(f"doc : {doc}")
    print(f"metadata : {meta}")
    print(f"embedding : {emb[:5]}")
    print(f"embedding dim : {len(emb)}")
    print("--------------------")

id : doc1
doc : 문서1:금요일 퇴근 후에 헬스장
metadata : {'tag': 'mes1'}
embedding : [-0.00911812 -0.02242651  0.03233086 -0.0236039   0.02917333]
embedding dim : 384
--------------------
id : doc2
doc : 문서2:잠은 언제 자나~
metadata : {'tag': 'mes2'}
embedding : [ 0.00403785  0.05862309  0.06415996 -0.04665814  0.03444706]
embedding dim : 384
--------------------


In [11]:
# 자료 수정 update - (sql)update
collection.update(
    ids=["doc2"],
    documents=['문서2:메세지 내용이 수정됨'],
    metadatas=[{'tag': 'edited-mes'}]
)

results = collection.get(include=['documents','metadatas','embeddings'])
for doc, meta, id, emb in zip(results['documents'], results['metadatas'], results['ids'], results['embeddings']):
    print(f"id : {id}")
    print(f"doc : {doc}")
    print(f"metadata : {meta}")
    print(f"embedding : {emb[:5]}")
    print(f"embedding dim : {len(emb)}")
    print("--------------------")

id : doc1
doc : 문서1:금요일 퇴근 후에 헬스장
metadata : {'tag': 'mes1'}
embedding : [-0.00911812 -0.02242651  0.03233086 -0.0236039   0.02917333]
embedding dim : 384
--------------------
id : doc2
doc : 문서2:메세지 내용이 수정됨
metadata : {'tag': 'edited-mes'}
embedding : [ 0.02002782  0.01221266  0.06960154 -0.01440814  0.04374016]
embedding dim : 384
--------------------


In [13]:
# 자료 삭제 delete - (sql)delete
collection.delete(ids=['doc1'])
collection.delete(where={'tag':'edited-mes'})


results = collection.get(include=['documents','metadatas','embeddings'])
for doc, meta, id, emb in zip(results['documents'], results['metadatas'], results['ids'], results['embeddings']):
    print(f"id : {id}")
    print(f"doc : {doc}")
    print(f"metadata : {meta}")
    print(f"embedding : {emb[:5]}")
    print(f"embedding dim : {len(emb)}")
    print("--------------------")